In [3]:
from sqlalchemy import create_engine
import pandas as pd

# Replace with your actual credentials
username = "root"
password = input("Enter your MySQL password: ")
host = "localhost"
database = "sakila"

# Create the connection string
engine = create_engine(f"mysql+pymysql://{username}:{password}@{host}/{database}")

# Test the connection by reading a small sample
df = pd.read_sql("SELECT title, release_year FROM film LIMIT 5;", engine)
print(df)

              title  release_year
0  ACADEMY DINOSAUR          2006
1    ACE GOLDFINGER          2006
2  ADAPTATION HOLES          2006
3  AFFAIR PREJUDICE          2006
4       AFRICAN EGG          2006


2-Write a Python function called rentals_month that retrieves rental data for a given month and year (passed as parameters) from the Sakila database as a Pandas DataFrame. The function should take in three parameters:

engine: an object representing the database connection engine to be used to establish a connection to the Sakila database.
month: an integer representing the month for which rental data is to be retrieved.
year: an integer representing the year for which rental data is to be retrieved.
The function should execute a SQL query to retrieve the rental data for the specified month and year from the rental table in the Sakila database, and return it as a pandas DataFrame.

In [14]:
def rentals_month(engine, month, year):
    """
    Retrieve rental data for a given month and year from the Sakila database.

    Parameters:
    engine: SQLAlchemy engine object for database connection.
    month: Integer representing the month (1-12).
    year: Integer representing the year (e.g., 2020).

    Returns:
    Pandas DataFrame containing rental data for the specified month and year.
    """
    query = f"""
    SELECT *
    FROM rental
    WHERE MONTH(rental_date) = {month} AND YEAR(rental_date) = {year};
    """
    
    df = pd.read_sql(query, engine)
    return df

rentals_month(engine, 5, 2005)




,rental_id,rental_date,inventory_id,customer_id,return_date,staff_id,last_update
0,1,2005-05-24 22:53:30,367,130,2005-05-26 22:04:30,1,2006-02-15 21:30:53
1,2,2005-05-24 22:54:33,1525,459,2005-05-28 19:40:33,1,2006-02-15 21:30:53
2,3,2005-05-24 23:03:39,1711,408,2005-06-01 22:12:39,1,2006-02-15 21:30:53
3,4,2005-05-24 23:04:41,2452,333,2005-06-03 01:43:41,2,2006-02-15 21:30:53
4,5,2005-05-24 23:05:21,2079,222,2005-06-02 04:33:21,1,2006-02-15 21:30:53
...,...,...,...,...,...,...,...
1151,1153,2005-05-31 21:36:44,2725,506,2005-06-10 01:26:44,2,2006-02-15 21:30:53
1152,1154,2005-05-31 21:42:09,2732,59,2005-06-08 16:40:09,1,2006-02-15 21:30:53
1153,1155,2005-05-31 22:17:11,2048,251,2005-06-04 20:27:11,2,2006-02-15 21:30:53
1154,1156,2005-05-31 22:37:34,460,106,2005-06-01 23:02:34,2,2006-02-15 21:30:53


Develop a Python function called rental_count_month that takes the DataFrame provided by rentals_month as input along with the month and year and returns a new DataFrame containing the number of rentals made by each customer_id during the selected month and year.

The function should also include the month and year as parameters and use them to name the new column according to the month and year, for example, if the input month is 05 and the year is 2005, the column name should be "rentals_05_2005".

In [22]:
def rental_count_month(df, month, year):
    if month<10:
        column_name = 'rentals_0'+ str(month)+'_'+str(year)
    else:           
        column_name = 'rentals_'+ str(month)+'_'+str(year)
    return df.groupby([df['customer_id']]).size().reset_index(name=column_name)


df = rentals_month(engine, 5, 2005)
rental_count_month (df, 5, 2005)

,customer_id,rentals_05_2005
0,1,2
1,2,1
2,3,2
3,5,3
4,6,3
...,...,...
515,594,4
516,595,1
517,596,6
518,597,2


Create a Python function called compare_rentals that takes two DataFrames as input containing the number of rentals made by each customer in different months and years. The function should return a combined DataFrame with a new 'difference' column, which is the difference between the number of rentals in the two months.

In [25]:
def compare_rentals (df1, df2):
    result = pd.merge(df1, df2, on='customer_id', how='outer').fillna(0)
    result['difference'] = result.iloc[:,1] - result.iloc[:,2]
    return result

df1 = rentals_month(engine, 5, 2005)
df1 = rental_count_month (df1, 5, 2005)

df2 = rentals_month(engine, 6, 2005)
df2 = rental_count_month (df2, 6, 2005)

compare_rentals (df1, df2)

,customer_id,rentals_05_2005,rentals_06_2005,difference
0,1,2.0,7.0,-5.0
1,2,1.0,1.0,0.0
2,3,2.0,4.0,-2.0
3,4,0.0,6.0,-6.0
4,5,3.0,5.0,-2.0
...,...,...,...,...
593,595,1.0,2.0,-1.0
594,596,6.0,2.0,4.0
595,597,2.0,3.0,-1.0
596,598,0.0,1.0,-1.0
